<a href="https://colab.research.google.com/github/comsadavid/AI-solutions/blob/main/competitions/mlcompete/Predictia_temperaturii_medii_zilnice.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, accuracy_score
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.neighbors import KNeighborsClassifier

from catboost import CatBoostClassifier
from catboost import CatBoostRegressor

from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [ ]:
!pip install catboost

In [ ]:
df_train = pd.read_csv("train.csv")
df_test = pd.read_csv('test.csv')

In [ ]:
df_train.head()

,SampleID,humidity,wind_speed,pressure,rainfall,cloud_cover,solar_radiation,day_of_year,temperature
0,1,52.472407,20.944851,1005.553988,5.190818,26.170568,679.263245,313,5.33
1,2,87.042858,16.082891,1016.257028,4.791819,24.697880,635.169456,328,3.43
2,3,73.919637,9.285828,1026.188375,0.256421,90.625458,501.470266,7,-1.81
3,4,65.919509,24.413851,1021.966747,3.412478,24.954620,769.233001,206,24.23
4,5,39.361118,20.541935,1024.196834,3.801956,27.194973,240.332161,114,13.53


In [ ]:
df_train.dtypes

,0
SampleID,int64
humidity,float64
wind_speed,float64
pressure,float64
rainfall,float64
cloud_cover,float64
solar_radiation,float64
day_of_year,int64
temperature,float64


In [ ]:
def impute_columns(df_train, df_test, columns, strategy):
    imputer = SimpleImputer(missing_values=np.nan, strategy=strategy)
    df_train[columns] = imputer.fit_transform(df_train[columns])
    df_test[columns] = imputer.transform(df_test[columns])

In [ ]:
numeric_columns = ['humidity', 'wind_speed', 'pressure', 'rainfall', 'cloud_cover', 'solar_radiation', 'day_of_year']
impute_columns(df_train, df_test, numeric_columns, 'mean')

In [ ]:
X = df_train.drop(['SampleID', 'temperature'], axis=1)
y = df_train['temperature']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
sc = StandardScaler()
X_train_scaled = sc.fit_transform(X_train)
X_test_scaled = sc.transform(X_test)

In [ ]:
def evaluate_model(model):
    y_train_pred = model.predict(X_train_scaled)
    y_test_pred = model.predict(X_test_scaled)

    print(f"Train {mean_absolute_error(y_train, y_train_pred)}, Test: {mean_absolute_error(y_test, y_test_pred)}")

In [ ]:
lr = LinearRegression()
lr.fit(X_train_scaled, y_train)
evaluate_model(lr)

Train 6.117498864046102, Test: 6.497459315951824


In [ ]:
rfr = RandomForestRegressor(n_estimators=300, min_samples_split=2, min_samples_leaf=2)
rfr.fit(X_train_scaled, y_train)
evaluate_model(rfr)

Train 0.7513827175775611, Test: 1.7042331273569025


In [ ]:
cbr = CatBoostRegressor()
cbr.fit(X_train_scaled, y_train)
evaluate_model(cbr)

Learning rate set to 0.035425
0:	learn: 7.8973529	total: 3.58ms	remaining: 3.58s
1:	learn: 7.7359589	total: 7.31ms	remaining: 3.65s
2:	learn: 7.5683332	total: 11.2ms	remaining: 3.73s
3:	learn: 7.4010371	total: 14.4ms	remaining: 3.59s
4:	learn: 7.2307012	total: 17.8ms	remaining: 3.54s
5:	learn: 7.0657495	total: 21.8ms	remaining: 3.6s
6:	learn: 6.9314130	total: 25.6ms	remaining: 3.64s
7:	learn: 6.7812441	total: 29ms	remaining: 3.6s
8:	learn: 6.6346792	total: 32.5ms	remaining: 3.58s
9:	learn: 6.4945209	total: 35.7ms	remaining: 3.54s
10:	learn: 6.3544804	total: 39.3ms	remaining: 3.53s
11:	learn: 6.2233302	total: 42.6ms	remaining: 3.51s
12:	learn: 6.0962606	total: 46.1ms	remaining: 3.5s
13:	learn: 5.9811810	total: 49.9ms	remaining: 3.52s
14:	learn: 5.8492165	total: 54.4ms	remaining: 3.57s
15:	learn: 5.7366032	total: 58.2ms	remaining: 3.58s
16:	learn: 5.6188923	total: 61.8ms	remaining: 3.57s
17:	learn: 5.5115899	total: 65.1ms	remaining: 3.55s
18:	learn: 5.4004627	total: 69.3ms	remaining: 3.5

In [ ]:
def solve(model):
    X_pred = df_test.drop(['SampleID'], axis=1)
    # Scale X_pred using the same StandardScaler fitted on the training data
    X_pred_scaled = sc.transform(X_pred)
    y_pred = model.predict(X_pred_scaled)

    output = pd.DataFrame({
        'SampleID':df_test['SampleID'],
        'temperature': y_pred
    })

    output.to_csv("submissions2.csv", index=False)

solve(cbr)